In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from apex import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers.optimization import AdamW
from source.version1.data import dataLoader
from source.version1.model import Model
from source.version1.train import trainModel
from source.version1.loss import Loss

In [3]:
def train(fold):
    loader = {}
    loader['batch_size'] = 2
    loader['fold'] = fold
    train, valid = dataLoader(**loader)
    model = Model()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.)
    model, optimizer = amp.initialize(model, optimizer, opt_level="O2", keep_batchnorm_fp32=True, verbosity=0)
    schedular = ReduceLROnPlateau(optimizer, factor=0.5, min_lr=1e-7, patience=1)
    trainer = {}
    trainer['model'] = model
    trainer['train'] = train
    trainer['valid'] = valid
    trainer['loss_fn'] = Loss(weight=0.)
    trainer['optimizer'] = optimizer
    trainer['save'] = '../../model/version-1/fold-{}/model.pt'.format(fold)
    trainer['epochs'] = 6
    trainer['batch'] = 2
    trainer['schedular'] = schedular
    trainModel(**trainer)
    model = model.cpu()
    del model
    return None

In [7]:
train(1)

100%|██████████| 4862/4862 [10:21<00:00,  7.82it/s, train_loss=0.3360, valid_loss=-0.3877]


In [8]:
train(2)

100%|██████████| 4862/4862 [10:26<00:00,  7.76it/s, train_loss=0.3362, valid_loss=-0.3834]


In [4]:
train(3)

100%|██████████| 4862/4862 [10:23<00:00,  7.79it/s, train_loss=0.3379, valid_loss=-0.3963]


In [5]:
train(4)

100%|██████████| 4862/4862 [10:26<00:00,  7.75it/s, train_loss=0.3408, valid_loss=-0.3864]


In [6]:
train(5)

100%|██████████| 4864/4864 [10:33<00:00,  7.68it/s, train_loss=0.3377, valid_loss=-0.3833]
